# Module 4 • Distributional Semantics and Word Embeddings

# Lesson 25 • Sentence and Document Embeddings

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 120–150 minutes

---

## Scope

This lesson moves from individual word vectors to representations of complete
sentences and documents. It covers pooling, weighting, similarity, retrieval,
clustering, classification, long-document handling, evaluation, and
multilingual considerations.

The notebook is self-contained and uses a small synthetic embedding space and
local datasets so every code cell runs without external downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain why word vectors must be composed into sentence vectors;
- implement sum, mean, maximum, and weighted pooling;
- explain length effects and normalization;
- construct TF-IDF-weighted sentence vectors;
- implement Smooth Inverse Frequency weighting;
- remove a dominant common direction;
- calculate sentence similarity;
- evaluate sentence embeddings with correlation;
- build embedding-based retrieval;
- cluster sentence vectors;
- compare dense and sparse text classifiers;
- analyze pooling failures and semantic limitations;
- handle long documents through chunking;
- discuss Arabic and multilingual sentence representations.

## Table of Contents

1. From Word Vectors to Text Vectors
2. Why Composition Is Difficult
3. Toy Embedding Space
4. Tokenization and Vector Lookup
5. Sum Pooling
6. Mean Pooling
7. Maximum Pooling
8. Normalization
9. TF-IDF-Weighted Pooling
10. Smooth Inverse Frequency
11. Removing the Common Direction
12. Sentence Similarity
13. Similarity Evaluation
14. Nearest-Neighbor Search
15. Semantic Retrieval
16. Clustering Sentence Embeddings
17. Visualizing Sentence Spaces
18. Document Classification
19. Comparing Dense and Sparse Representations
20. Error Analysis
21. Supervised Sentence Representations
22. Pair Representations
23. Long Documents and Chunking
24. Static Versus Contextual Sentence Embeddings
25. Common Failure Modes
26. Bias and Responsible Use
27. Arabic and Multilingual Considerations
28. Reproducibility and Reporting
29. Knowledge Check
30. Exercises
31. Summary and Next Module

# 1. From Word Vectors to Text Vectors

A word embedding represents one vocabulary item. Many NLP tasks require one
vector for a longer unit:

- phrase;
- sentence;
- paragraph;
- document;
- conversation;
- query.

In [ ]:
import hashlib
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

composition_tasks = pd.DataFrame(
    [
        ("Sentence similarity", "sentence"),
        ("Document classification", "document"),
        ("Semantic search", "query and document"),
        ("Clustering", "sentence or document"),
        ("Duplicate detection", "sentence pair"),
    ],
    columns=["Task", "Required representation"],
)

composition_tasks

A sentence vector should ideally capture enough information for the downstream
task while remaining computationally manageable.

# 2. Why Composition Is Difficult

A sentence is more than an unordered collection of words.

Consider:

```text
The dog chased the cat.
The cat chased the dog.
```

Mean pooling produces the same vector for both sentences because the words are
identical.

In [ ]:
composition_challenges = pd.DataFrame(
    [
        ("Word order", "mean pooling ignores order"),
        ("Negation", "not good may remain close to good"),
        ("Polysemy", "one static vector mixes senses"),
        ("Syntax", "subject and object roles are lost"),
        ("Length", "sum vectors grow with document length"),
        ("Common words", "frequent terms may dominate"),
    ],
    columns=["Challenge", "Effect"],
)

composition_challenges

Pooling is useful as a baseline, but its assumptions must be explicit.

# 3. Toy Embedding Space

We construct a small semantic space with six broad domains:

- health;
- finance;
- technology;
- travel;
- education;
- environment.

In [ ]:
EMBEDDING_DIMENSION = 20
RANDOM_SEED = 42
generator = np.random.default_rng(RANDOM_SEED)

domain_words = {
    "health": [
        "doctor", "nurse", "patient", "hospital", "clinic",
        "medicine", "treatment", "health", "exercise", "nutrition",
        "diagnosis", "medical",
    ],
    "finance": [
        "bank", "loan", "payment", "invoice", "money", "interest",
        "refund", "billing", "account", "card", "charge", "price",
    ],
    "technology": [
        "software", "application", "server", "network", "computer",
        "device", "system", "data", "error", "update", "install",
        "upload",
    ],
    "travel": [
        "flight", "airport", "hotel", "travel", "tourist", "ticket",
        "luggage", "beach", "museum", "city", "reservation", "journey",
    ],
    "education": [
        "teacher", "student", "school", "university", "lesson",
        "course", "research", "professor", "study", "class",
        "learning", "exam",
    ],
    "environment": [
        "climate", "forest", "water", "energy", "pollution",
        "environment", "recycling", "carbon", "ocean", "wildlife",
        "solar", "temperature",
    ],
}

shared_words = [
    "need", "problem", "help", "please", "today", "service",
    "information", "request", "important", "new", "good", "bad",
]

domain_centers = {
    domain: generator.normal(
        0.0,
        1.0,
        size=EMBEDDING_DIMENSION,
    )
    for domain in domain_words
}

word_vectors = {}

for domain, words in domain_words.items():
    center = domain_centers[domain]

    for word in words:
        word_vectors[word] = (
            center
            + generator.normal(
                0.0,
                0.18,
                size=EMBEDDING_DIMENSION,
            )
        )

for word in shared_words:
    word_vectors[word] = generator.normal(
        0.0,
        0.35,
        size=EMBEDDING_DIMENSION,
    )

print("Embedding vocabulary:", len(word_vectors))

The synthetic vectors create controlled semantic neighborhoods for teaching.
Real pretrained vectors are learned from corpora.

# 4. Tokenization and Vector Lookup

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


def deterministic_oov_vector(
    word: str,
    dimension: int = EMBEDDING_DIMENSION,
) -> np.ndarray:
    marked = f"<{word}>"
    ngrams = []

    for n in range(3, 6):
        for start in range(
            len(marked) - n + 1
        ):
            ngrams.append(
                marked[start:start + n]
            )

    if not ngrams:
        return np.zeros(dimension)

    vectors = []

    for ngram in ngrams:
        digest = hashlib.sha256(
            ngram.encode("utf-8")
        ).digest()

        seed = int.from_bytes(
            digest[:8],
            byteorder="little",
            signed=False,
        )

        local_generator = np.random.default_rng(
            seed
        )

        vectors.append(
            local_generator.normal(
                0.0,
                0.08,
                size=dimension,
            )
        )

    return np.mean(vectors, axis=0)


def lookup_vector(
    word: str,
    use_oov: bool = True,
) -> np.ndarray:
    if word in word_vectors:
        return word_vectors[word]

    if use_oov:
        return deterministic_oov_vector(word)

    return np.zeros(EMBEDDING_DIMENSION)


tokenize(
    "The doctor reviewed the patient's diagnosis."
)

A deterministic subword fallback provides a vector for unseen forms, but it is
not a trained semantic representation.

# 5. Sum Pooling

Sum pooling adds token vectors:

\[
s = \sum_{i=1}^{n} v_i
\]

In [ ]:
def sum_pool(
    text: str,
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(EMBEDDING_DIMENSION)

    return np.sum(
        [
            lookup_vector(token)
            for token in tokens
        ],
        axis=0,
    )


short_sum = sum_pool("doctor patient")
long_sum = sum_pool(
    "doctor patient hospital medicine treatment"
)

print("Short norm:", round(np.linalg.norm(short_sum), 3))
print("Long norm:", round(np.linalg.norm(long_sum), 3))

Sum vectors are affected by sentence length because adding more token vectors
usually increases vector magnitude.

# 6. Mean Pooling

Mean pooling divides by the number of tokens:

\[
s = \frac{1}{n}\sum_{i=1}^{n} v_i
\]

In [ ]:
def mean_pool(
    text: str,
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(EMBEDDING_DIMENSION)

    matrix = np.vstack(
        [
            lookup_vector(token)
            for token in tokens
        ]
    )

    return matrix.mean(axis=0)


short_mean = mean_pool("doctor patient")
long_mean = mean_pool(
    "doctor patient hospital medicine treatment"
)

print("Short norm:", round(np.linalg.norm(short_mean), 3))
print("Long norm:", round(np.linalg.norm(long_mean), 3))

Mean pooling reduces direct length effects but still ignores order and syntax.

# 7. Maximum Pooling

Maximum pooling keeps the largest value in each dimension:

\[
s_j = \max_i v_{ij}
\]

In [ ]:
def max_pool(
    text: str,
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(EMBEDDING_DIMENSION)

    matrix = np.vstack(
        [
            lookup_vector(token)
            for token in tokens
        ]
    )

    return matrix.max(axis=0)


max_pool(
    "doctor nurse patient hospital"
)[:8]

Maximum pooling can preserve strong features but may overemphasize one token and
discard frequency information.

# 8. Normalization

L2 normalization converts a vector to unit length:

\[
\hat{s} = \frac{s}{||s||}
\]

In [ ]:
def l2_normalize(
    vector: np.ndarray,
) -> np.ndarray:
    norm = np.linalg.norm(vector)

    if norm == 0:
        return vector.copy()

    return vector / norm


normalized = l2_normalize(
    mean_pool(
        "doctor patient hospital"
    )
)

print("Normalized norm:", np.linalg.norm(normalized))

Normalization is common before cosine similarity and nearest-neighbor search.

# 9. TF-IDF-Weighted Pooling

Equal weighting may allow frequent or generic words to dominate.

TF-IDF-weighted pooling uses:

\[
s =
\frac{\sum_i weight_i v_i}
{\sum_i weight_i}
\]

In [ ]:
similarity_corpus = [
    "doctor treats patient in hospital",
    "nurse cares for patient in clinic",
    "bank processes payment and refund",
    "invoice contains a billing charge",
    "software update caused server error",
    "application cannot connect to network",
    "tourist booked hotel and flight",
    "airport lost the passenger luggage",
    "teacher explains lesson to student",
    "professor leads research at university",
    "solar energy reduces carbon pollution",
    "forest and ocean support wildlife",
]

tfidf_weighting = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
)

tfidf_weighting.fit(similarity_corpus)

tfidf_vocabulary = tfidf_weighting.vocabulary_
idf_values = tfidf_weighting.idf_


def tfidf_weighted_pool(
    text: str,
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(
            EMBEDDING_DIMENSION
        )

    token_counts = Counter(tokens)
    weighted_vectors = []
    weights = []

    for token, count in token_counts.items():
        feature_id = tfidf_vocabulary.get(
            token
        )

        idf = (
            float(idf_values[feature_id])
            if feature_id is not None
            else 1.0
        )

        tf = count / len(tokens)
        weight = tf * idf

        weighted_vectors.append(
            weight * lookup_vector(token)
        )
        weights.append(weight)

    return (
        np.sum(weighted_vectors, axis=0)
        / max(sum(weights), 1e-12)
    )


tfidf_weighted_pool(
    "doctor patient hospital"
)[:8]

The weighting model must be fitted only on training or reference data, not on a
held-out evaluation set.

# 10. Smooth Inverse Frequency

Smooth Inverse Frequency (SIF) assigns a word with estimated probability
\(p(w)\) the weight:

\[
\frac{a}{a + p(w)}
\]

where \(a\) is a small smoothing constant.

In [ ]:
corpus_token_counts = Counter(
    token
    for text in similarity_corpus
    for token in tokenize(text)
)

total_corpus_tokens = sum(
    corpus_token_counts.values()
)

word_probabilities = {
    word: count / total_corpus_tokens
    for word, count in corpus_token_counts.items()
}


def sif_pool(
    text: str,
    smoothing: float = 1e-3,
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(
            EMBEDDING_DIMENSION
        )

    weighted_vectors = []
    weights = []

    for token in tokens:
        probability = word_probabilities.get(
            token,
            1.0 / max(
                total_corpus_tokens,
                1,
            ),
        )

        weight = (
            smoothing
            / (
                smoothing
                + probability
            )
        )

        weighted_vectors.append(
            weight * lookup_vector(token)
        )
        weights.append(weight)

    return (
        np.sum(weighted_vectors, axis=0)
        / max(sum(weights), 1e-12)
    )


sif_pool(
    "doctor treats patient in hospital"
)[:8]

SIF down-weights frequent words without requiring class labels.

# 11. Removing the Common Direction

SIF commonly removes the first principal component from sentence vectors. This
attempts to subtract a dominant corpus-wide direction.

In [ ]:
raw_sif_matrix = np.vstack(
    [
        sif_pool(text)
        for text in similarity_corpus
    ]
)

common_direction_model = PCA(
    n_components=1,
    random_state=42,
)

common_direction_model.fit(
    raw_sif_matrix
)

common_direction = (
    common_direction_model.components_[0]
)


def remove_common_direction(
    vector: np.ndarray,
    direction: np.ndarray,
) -> np.ndarray:
    return (
        vector
        - np.dot(vector, direction)
        * direction
    )


sif_corrected_matrix = np.vstack(
    [
        remove_common_direction(
            vector,
            common_direction,
        )
        for vector in raw_sif_matrix
    ]
)

print("Matrix shape:", sif_corrected_matrix.shape)

Principal-component removal may improve some datasets and harm others. It should
be evaluated, not assumed beneficial.

# 12. Sentence Similarity

Cosine similarity compares two sentence vectors:

\[
cosine(a,b) =
\frac{a \cdot b}
{||a||\,||b||}
\]

In [ ]:
def cosine_between(
    left: np.ndarray,
    right: np.ndarray,
) -> float:
    denominator = (
        np.linalg.norm(left)
        * np.linalg.norm(right)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(left, right)
        / denominator
    )


sentence_pairs = [
    (
        "doctor treats patient in hospital",
        "nurse cares for patient in clinic",
    ),
    (
        "doctor treats patient in hospital",
        "bank processes payment and refund",
    ),
    (
        "teacher explains lesson to student",
        "professor leads research at university",
    ),
]

for left, right in sentence_pairs:
    score = cosine_between(
        mean_pool(left),
        mean_pool(right),
    )

    print(f"{score:.3f} | {left} | {right}")

Similarity may reflect topic, lexical overlap, or domain rather than full
sentence meaning.

# 13. Similarity Evaluation

We create a small human-scored semantic similarity set. Scores range from 0 to
5.

In [ ]:
similarity_examples = pd.DataFrame(
    [
        (
            "doctor treats patient in hospital",
            "nurse cares for patient in clinic",
            4.6,
        ),
        (
            "bank processes payment and refund",
            "invoice contains a billing charge",
            4.4,
        ),
        (
            "software update caused server error",
            "application cannot connect to network",
            4.1,
        ),
        (
            "tourist booked hotel and flight",
            "airport lost the passenger luggage",
            3.7,
        ),
        (
            "teacher explains lesson to student",
            "professor leads research at university",
            4.0,
        ),
        (
            "solar energy reduces carbon pollution",
            "forest and ocean support wildlife",
            3.8,
        ),
        (
            "doctor treats patient in hospital",
            "invoice contains a billing charge",
            0.5,
        ),
        (
            "software update caused server error",
            "tourist booked hotel and flight",
            0.4,
        ),
        (
            "bank processes payment and refund",
            "teacher explains lesson to student",
            0.3,
        ),
        (
            "forest and ocean support wildlife",
            "application cannot connect to network",
            0.2,
        ),
    ],
    columns=[
        "sentence_1",
        "sentence_2",
        "human_score",
    ],
)

similarity_examples

In [ ]:
pooling_methods = {
    "mean": mean_pool,
    "max": max_pool,
    "tfidf": tfidf_weighted_pool,
    "sif": sif_pool,
}

evaluation_rows = []

for method_name, method in pooling_methods.items():
    model_scores = []

    for row in similarity_examples.itertuples(
        index=False
    ):
        model_scores.append(
            cosine_between(
                method(row.sentence_1),
                method(row.sentence_2),
            )
        )

    correlation, p_value = spearmanr(
        similarity_examples["human_score"],
        model_scores,
    )

    evaluation_rows.append(
        {
            "method": method_name,
            "spearman_correlation": correlation,
            "p_value": p_value,
        }
    )

similarity_evaluation = pd.DataFrame(
    evaluation_rows
).sort_values(
    "spearman_correlation",
    ascending=False,
)

similarity_evaluation.round(3)

The example set is too small for formal conclusions. Reliable evaluation uses
established datasets, confidence intervals, and multiple domains.

# 14. Nearest-Neighbor Search

In [ ]:
indexed_sentences = similarity_corpus
indexed_matrix = np.vstack(
    [
        tfidf_weighted_pool(text)
        for text in indexed_sentences
    ]
)

indexed_matrix_normalized = np.vstack(
    [
        l2_normalize(vector)
        for vector in indexed_matrix
    ]
)


def sentence_neighbors(
    query: str,
    top_k: int = 4,
) -> pd.DataFrame:
    query_vector = l2_normalize(
        tfidf_weighted_pool(query)
    )

    scores = (
        indexed_matrix_normalized
        @ query_vector
    )

    return (
        pd.DataFrame(
            {
                "sentence": indexed_sentences,
                "similarity": scores,
            }
        )
        .sort_values(
            ["similarity", "sentence"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


sentence_neighbors(
    "medical staff care for a patient"
)

# 15. Semantic Retrieval

Sentence embeddings can retrieve semantically related documents even without
exact lexical overlap.

In [ ]:
retrieval_queries = {
    "medical care": {0, 1},
    "financial billing": {2, 3},
    "computer connection problem": {4, 5},
    "holiday transport": {6, 7},
    "academic teaching": {8, 9},
    "environment protection": {10, 11},
}


def reciprocal_rank(
    ranked_indices: list[int],
    relevant_indices: set[int],
) -> float:
    for rank, index in enumerate(
        ranked_indices,
        start=1,
    ):
        if index in relevant_indices:
            return 1.0 / rank

    return 0.0


retrieval_rows = []

for query, relevant in retrieval_queries.items():
    query_vector = l2_normalize(
        tfidf_weighted_pool(query)
    )

    scores = (
        indexed_matrix_normalized
        @ query_vector
    )

    ranking = np.argsort(scores)[::-1].tolist()

    retrieval_rows.append(
        {
            "query": query,
            "top_sentence": indexed_sentences[
                ranking[0]
            ],
            "reciprocal_rank": reciprocal_rank(
                ranking,
                relevant,
            ),
        }
    )

retrieval_evaluation = pd.DataFrame(
    retrieval_rows
)

retrieval_evaluation

In [ ]:
print(
    "Mean Reciprocal Rank:",
    round(
        retrieval_evaluation[
            "reciprocal_rank"
        ].mean(),
        3,
    ),
)

Retrieval evaluation requires explicit relevance judgments.

# 16. Clustering Sentence Embeddings

In [ ]:
cluster_sentences = [
    "doctor treats patient",
    "nurse works in hospital",
    "medical diagnosis needs treatment",
    "bank approves loan",
    "invoice contains payment charge",
    "card refund request",
    "software update caused error",
    "server network unavailable",
    "install application on computer",
    "tourist booked hotel",
    "flight arrived at airport",
    "travel ticket and luggage",
]

cluster_vectors = np.vstack(
    [
        tfidf_weighted_pool(text)
        for text in cluster_sentences
    ]
)

cluster_model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20,
)

cluster_labels = cluster_model.fit_predict(
    cluster_vectors
)

pd.DataFrame(
    {
        "sentence": cluster_sentences,
        "cluster": cluster_labels,
    }
).sort_values(
    ["cluster", "sentence"]
)

In [ ]:
cluster_silhouette = silhouette_score(
    cluster_vectors,
    cluster_labels,
    metric="cosine",
)

print(
    "Cosine silhouette score:",
    round(cluster_silhouette, 3),
)

Cluster numbers are arbitrary. Human inspection is required to interpret
cluster themes.

# 17. Visualizing Sentence Spaces

PCA projects vectors into two dimensions for exploration.

In [ ]:
projection_model = PCA(
    n_components=2,
    random_state=42,
)

projected_vectors = projection_model.fit_transform(
    cluster_vectors
)

plt.figure(figsize=(9, 6))
plt.scatter(
    projected_vectors[:, 0],
    projected_vectors[:, 1],
)

for index, sentence in enumerate(
    cluster_sentences
):
    short_label = " ".join(
        sentence.split()[:3]
    )

    plt.text(
        projected_vectors[index, 0],
        projected_vectors[index, 1],
        short_label,
    )

plt.title("Sentence Embeddings Projected with PCA")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.tight_layout()
plt.show()

A two-dimensional plot distorts the original geometry and should not replace
quantitative evaluation.

# 18. Document Classification

We compare dense sentence vectors with a sparse TF-IDF baseline.

In [ ]:
classification_records = [
    ("The doctor reviewed the diagnosis", "health"),
    ("The nurse treated the patient", "health"),
    ("Exercise improves medical health", "health"),
    ("The hospital provides treatment", "health"),
    ("Nutrition supports patient recovery", "health"),
    ("The clinic needs medicine", "health"),
    ("The bank approved the loan", "finance"),
    ("The invoice contains a charge", "finance"),
    ("The card payment failed", "finance"),
    ("The refund request is important", "finance"),
    ("Billing changed the account price", "finance"),
    ("The bank increased interest", "finance"),
    ("The application caused an error", "technology"),
    ("The server lost network access", "technology"),
    ("Please install the software update", "technology"),
    ("The computer cannot upload data", "technology"),
    ("The device system needs help", "technology"),
    ("The application cannot connect", "technology"),
    ("The tourist booked a hotel", "travel"),
    ("The flight arrived at the airport", "travel"),
    ("The luggage was lost", "travel"),
    ("The travel ticket changed", "travel"),
    ("The beach journey starts today", "travel"),
    ("The museum is in the city", "travel"),
]

classification_data = pd.DataFrame(
    classification_records,
    columns=["text", "label"],
)

X_train, X_test, y_train, y_test = (
    train_test_split(
        classification_data["text"],
        classification_data["label"],
        test_size=0.33,
        random_state=42,
        stratify=classification_data["label"],
    )
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

In [ ]:
dense_train = np.vstack(
    [
        tfidf_weighted_pool(text)
        for text in X_train
    ]
)

dense_test = np.vstack(
    [
        tfidf_weighted_pool(text)
        for text in X_test
    ]
)

dense_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

dense_classifier.fit(
    dense_train,
    y_train,
)

dense_predictions = dense_classifier.predict(
    dense_test
)

dense_macro_f1 = f1_score(
    y_test,
    dense_predictions,
    average="macro",
)

print(
    "Dense embedding macro F1:",
    round(dense_macro_f1, 3),
)

# 19. Comparing Dense and Sparse Representations

In [ ]:
sparse_classifier = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                tokenizer=tokenize,
                token_pattern=None,
                lowercase=False,
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

sparse_classifier.fit(
    X_train,
    y_train,
)

sparse_predictions = sparse_classifier.predict(
    X_test
)

sparse_macro_f1 = f1_score(
    y_test,
    sparse_predictions,
    average="macro",
)

pd.DataFrame(
    [
        (
            "Dense weighted embedding",
            dense_macro_f1,
        ),
        (
            "Sparse TF-IDF",
            sparse_macro_f1,
        ),
    ],
    columns=["Representation", "Macro F1"],
)

Dense embeddings may generalize across related words. Sparse TF-IDF can preserve
exact lexical distinctions.

# 20. Error Analysis

In [ ]:
classification_results = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual": y_test.reset_index(drop=True),
        "dense_prediction": dense_predictions,
        "sparse_prediction": sparse_predictions,
    }
)

classification_results["dense_correct"] = (
    classification_results["actual"]
    == classification_results[
        "dense_prediction"
    ]
)

classification_results["sparse_correct"] = (
    classification_results["actual"]
    == classification_results[
        "sparse_prediction"
    ]
)

classification_results

In [ ]:
class_names = sorted(
    classification_data["label"].unique()
)

dense_confusion = confusion_matrix(
    y_test,
    dense_predictions,
    labels=class_names,
)

pd.DataFrame(
    dense_confusion,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

Review failures for OOV tokens, mixed topics, short text, negation, and
misleading semantic neighbors.

# 21. Supervised Sentence Representations

A classifier trained on pooled vectors learns task-specific decision
directions. A deeper model can also learn a task-specific projection before
classification.

In [ ]:
projection_dimension = 8
projection_generator = np.random.default_rng(42)

supervised_projection = (
    projection_generator.normal(
        0.0,
        0.2,
        size=(
            EMBEDDING_DIMENSION,
            projection_dimension,
        ),
    )
)

projected_train = np.tanh(
    dense_train
    @ supervised_projection
)

projected_test = np.tanh(
    dense_test
    @ supervised_projection
)

projected_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

projected_classifier.fit(
    projected_train,
    y_train,
)

projected_predictions = (
    projected_classifier.predict(
        projected_test
    )
)

print(
    "Random-projection baseline macro F1:",
    round(
        f1_score(
            y_test,
            projected_predictions,
            average="macro",
        ),
        3,
    ),
)

The projection above is random and fixed. Neural sentence encoders learn the
projection jointly with the task objective.

# 22. Pair Representations

Sentence-pair models often combine vectors using:

- concatenation;
- absolute difference;
- elementwise product;
- cosine similarity.

In [ ]:
left_vector = mean_pool(
    "doctor treats patient"
)
right_vector = mean_pool(
    "nurse cares for patient"
)

pair_features = np.concatenate(
    [
        left_vector,
        right_vector,
        np.abs(
            left_vector
            - right_vector
        ),
        left_vector
        * right_vector,
        np.array(
            [
                cosine_between(
                    left_vector,
                    right_vector,
                )
            ]
        ),
    ]
)

print(
    "Pair-feature dimension:",
    pair_features.shape[0],
)

Pair features support paraphrase detection, duplicate detection, and semantic
relation classification.

# 23. Long Documents and Chunking

Averaging an entire long document may blur several topics.

A practical alternative:

1. split the document into chunks;
2. embed each chunk;
3. aggregate chunk vectors;
4. preserve chunk-level scores when useful.

In [ ]:
long_document = (
    "The hospital introduced a new medical system. "
    "Doctors and nurses reviewed patient treatment. "
    "The software server later produced a network error. "
    "The technology team installed an update."
)


def split_into_chunks(
    text: str,
    chunk_size: int = 8,
) -> list[str]:
    tokens = tokenize(text)

    return [
        " ".join(
            tokens[start:start + chunk_size]
        )
        for start in range(
            0,
            len(tokens),
            chunk_size,
        )
    ]


chunks = split_into_chunks(
    long_document,
    chunk_size=8,
)

chunk_vectors = np.vstack(
    [
        tfidf_weighted_pool(chunk)
        for chunk in chunks
    ]
)

chunk_frame = pd.DataFrame(
    {
        "chunk": chunks,
        "vector_norm": np.linalg.norm(
            chunk_vectors,
            axis=1,
        ),
    }
)

chunk_frame

Chunking preserves local themes but introduces decisions about chunk boundaries,
overlap, and final aggregation.

# 24. Static Versus Contextual Sentence Embeddings

Static pooling combines fixed word vectors.

Contextual sentence encoders use surrounding tokens and model parameters to
generate occurrence-specific representations.

In [ ]:
static_contextual = pd.DataFrame(
    [
        (
            "Static pooling",
            "fixed word vectors",
            "fast and interpretable",
            "weak syntax and polysemy handling",
        ),
        (
            "Contextual encoder",
            "context-specific token vectors",
            "strong compositional modeling",
            "greater compute and complexity",
        ),
    ],
    columns=[
        "Approach",
        "Input representation",
        "Strength",
        "Limitation",
    ],
)

static_contextual

Contextual sentence embeddings will be studied after neural and Transformer
foundations.

# 25. Common Failure Modes

- word order ignored;
- negation lost;
- rare words poorly represented;
- OOV fallback lacks semantics;
- long documents over-compressed;
- mixed-topic documents blurred;
- common direction dominates;
- cosine similarity reflects topic instead of meaning;
- anisotropic spaces produce overly high similarities.

In [ ]:
failure_modes = pd.DataFrame(
    [
        ("Order loss", "use sequence or contextual encoders"),
        ("Negation failure", "add contextual modeling"),
        ("Long-document blur", "use chunking or hierarchy"),
        ("OOV weakness", "use trained subword representations"),
        ("Domain mismatch", "adapt embeddings to the domain"),
        ("Anisotropy", "center, normalize, or remove directions"),
    ],
    columns=["Failure", "Possible response"],
)

failure_modes

# 26. Bias and Responsible Use

Sentence vectors inherit associations from their word vectors and corpora.

Bias evaluation should inspect:

- demographic substitutions;
- dialectal variation;
- geographic references;
- sentence-pair similarity disparities;
- retrieval ranking disparities;
- downstream class errors.

In [ ]:
bias_audit = pd.DataFrame(
    [
        ("Similarity", "Do demographic substitutions change scores?"),
        ("Retrieval", "Are some names ranked differently?"),
        ("Coverage", "Which dialects produce more OOV tokens?"),
        ("Classification", "Do errors differ by language variety?"),
        ("Documentation", "Are corpus and limits reported?"),
    ],
    columns=["Audit area", "Question"],
)

bias_audit

Pooling can hide which words caused a biased or incorrect result, so token-level
inspection remains important.

# 27. Arabic and Multilingual Considerations

Arabic sentence embeddings are affected by:

- clitic attachment;
- rich morphology;
- optional diacritics;
- orthographic variants;
- MSA and dialect;
- code-switching;
- tokenization policy.

In [ ]:
arabic_domain_centers = {
    "health": np.array(
        [1.0, 0.9, 0.1, 0.0, 0.1, 0.0]
    ),
    "finance": np.array(
        [0.0, 0.1, 1.0, 0.9, 0.0, 0.1]
    ),
}

arabic_vectors = {
    "طبيب": (
        arabic_domain_centers["health"]
        + np.array(
            [0.05, -0.02, 0, 0, 0, 0]
        )
    ),
    "ممرض": (
        arabic_domain_centers["health"]
        + np.array(
            [-0.03, 0.04, 0, 0, 0, 0]
        )
    ),
    "مريض": (
        arabic_domain_centers["health"]
        + np.array(
            [0.02, 0.02, 0, 0, 0, 0]
        )
    ),
    "مستشفى": (
        arabic_domain_centers["health"]
        + np.array(
            [0.01, -0.01, 0, 0, 0, 0]
        )
    ),
    "بنك": (
        arabic_domain_centers["finance"]
        + np.array(
            [0, 0, 0.04, -0.02, 0, 0]
        )
    ),
    "دفع": (
        arabic_domain_centers["finance"]
        + np.array(
            [0, 0, -0.01, 0.04, 0, 0]
        )
    ),
    "فاتورة": (
        arabic_domain_centers["finance"]
        + np.array(
            [0, 0, 0.03, 0.01, 0, 0]
        )
    ),
}


def arabic_mean_pool(
    text: str,
) -> np.ndarray:
    tokens = text.split()
    covered = [
        arabic_vectors[token]
        for token in tokens
        if token in arabic_vectors
    ]

    if not covered:
        return np.zeros(6)

    return np.mean(
        covered,
        axis=0,
    )


arabic_health_1 = arabic_mean_pool(
    "طبيب يعالج مريض"
)
arabic_health_2 = arabic_mean_pool(
    "ممرض يعمل في مستشفى"
)
arabic_finance = arabic_mean_pool(
    "بنك يراجع فاتورة دفع"
)

print(
    "Health similarity:",
    round(
        cosine_between(
            arabic_health_1,
            arabic_health_2,
        ),
        3,
    ),
)
print(
    "Health-finance similarity:",
    round(
        cosine_between(
            arabic_health_1,
            arabic_finance,
        ),
        3,
    ),
)

This example omits attached clitics and spelling variants. Real Arabic systems
require an explicit normalization and segmentation policy.

## 27.1 Cross-Lingual Sentence Spaces

Multilingual sentence embeddings place several languages in a shared vector
space so translations or semantically equivalent sentences are close.

In [ ]:
cross_lingual_examples = pd.DataFrame(
    [
        (
            "The doctor treats the patient",
            "الطبيب يعالج المريض",
            "translation pair",
        ),
        (
            "The bank processes the payment",
            "البنك يعالج عملية الدفع",
            "translation pair",
        ),
    ],
    columns=[
        "English",
        "Arabic",
        "Relation",
    ],
)

cross_lingual_examples

Cross-lingual alignment quality depends on language coverage, parallel data,
scripts, domains, and evaluation design.

# 28. Reproducibility and Reporting

Report:

- word-embedding source;
- vector dimension;
- tokenizer;
- OOV strategy;
- pooling method;
- weighting corpus;
- normalization;
- common-direction removal;
- sentence or document length;
- evaluation dataset;
- random seeds;
- downstream model and metrics.

In [ ]:
import platform
import sklearn

metadata = pd.Series(
    {
        "embedding_source": "synthetic offline space",
        "embedding_dimension": EMBEDDING_DIMENSION,
        "embedding_vocabulary": len(word_vectors),
        "pooling_methods": "sum, mean, max, TF-IDF, SIF",
        "oov_strategy": "deterministic character n-grams",
        "random_seed": RANDOM_SEED,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "scikit_learn_version": sklearn.__version__,
    },
    name="Sentence embedding experiment",
)

metadata

Reproducibility requires the weighting corpus and preprocessing configuration,
not only the final sentence vectors.

# 29. Knowledge Check

1. Why must word vectors be composed into sentence vectors?
2. How do sum and mean pooling differ?
3. What does maximum pooling retain?
4. Why normalize sentence vectors?
5. How does TF-IDF weighting affect pooling?
6. What is Smooth Inverse Frequency?
7. Why remove a common principal direction?
8. What does cosine similarity measure?
9. Why can similar topics produce high similarity without equivalent meaning?
10. How are sentence vectors used in retrieval?
11. Why are cluster labels arbitrary?
12. Why compare dense embeddings with sparse TF-IDF?
13. What information do pairwise difference and product features provide?
14. Why is chunking useful for long documents?
15. Which Arabic properties affect sentence embeddings?

# 30. Exercises

## Exercise 1 — Pooling

Compare sum, mean, maximum, and minimum pooling.

## Exercise 2 — Weighting

Compare uniform, TF-IDF, and SIF weights.

## Exercise 3 — Common Direction

Remove one, two, and three principal directions and evaluate similarity.

## Exercise 4 — Semantic Similarity

Build a larger human-scored sentence-pair dataset.

## Exercise 5 — Retrieval

Evaluate Precision@k, Recall@k, MRR, and nDCG.

## Exercise 6 — Classification

Compare dense embeddings with word and character TF-IDF.

## Exercise 7 — Long Documents

Compare full-document pooling with overlapping chunks.

## Exercise 8 — Arabic Sentences

Compare raw, normalized, and segmented Arabic sentence vectors.

## Challenge Exercises

1. Implement attention-weighted pooling.
2. Learn a supervised projection with gradient descent.
3. Add sentence-pair classification.
4. Implement hierarchical document pooling.
5. Build a multilingual semantic-search evaluation.

# 31. Summary and Next Module

In this lesson:

- word vectors were composed into sentence and document representations;
- sum, mean, and maximum pooling were implemented;
- normalization reduced magnitude effects;
- TF-IDF and SIF weighting reduced generic-word influence;
- common-direction removal was demonstrated;
- sentence similarity was evaluated with Spearman correlation;
- sentence embeddings supported nearest-neighbor search and retrieval;
- clustering and PCA provided corpus-level exploration;
- dense classification was compared with sparse TF-IDF;
- pair features supported sentence-relation tasks;
- chunking preserved local themes in long documents;
- static pooling limitations motivated contextual encoders;
- Arabic and multilingual sentence representation challenges were analyzed.

## Next Module

**Module 5: Neural Networks for Natural Language Processing**

**Lesson 26: Neural Network Foundations for NLP** introduces tensors, dense
layers, activation functions, losses, gradient descent, batching, and the role
of embedding layers in neural text models.

# References

- Arora, S., Liang, Y., & Ma, T. *A Simple but Tough-to-Beat Baseline for Sentence Embeddings*.
- Le, Q., & Mikolov, T. *Distributed Representations of Sentences and Documents*.
- Reimers, N., & Gurevych, I. sentence-embedding literature.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- semantic textual similarity, retrieval, pooling, and multilingual embedding literature.